# Toehold switch — single input

Manual test harness for `engine.gates.toehold.ToeholdGate`. This family is
**implemented** (`available = True`), real ViennaRNA folding throughout.

Unlike `AntisenseNotGate`, this gate takes no payload argument — the switch's own
hairpin does not need to know the downstream gene's sequence, only that translation
should start at the `AUG` this gate places at the very end of its emitted sequence.
A caller downstream (`PlasmidBuilder`) fuses the real payload after it.

## The mechanism

A toehold switch is an mRNA that will not translate itself until told to. It folds
into a hairpin: the ribosome binding site sits in the accessible **loop**, the
**start codon** is buried in the **stem**, and a single-stranded **toehold** hangs
off the 5' end. The trigger pairs with the toehold, unzips the stem, frees the start
codon, and translation begins.

A good design keeps the OFF hairpin dark (low leakage) while making trigger binding
more favourable still (high dynamic range). Those pull against each other — which is
why design is a search, not a formula.

## Setup

In [ ]:
# Put the shared _fixtures.py on the path. It lives in the notebooks/ root, one
# level up from this gate's folder; search upward so the notebook works wherever
# Jupyter is launched. fx.bootstrap() then adds <repo>/src so `import engine...`
# resolves. No Django, no worker, no pipeline.
import sys, pathlib

for _base in (pathlib.Path.cwd(), *pathlib.Path.cwd().parents):
    if (_base / '_fixtures.py').exists():
        if str(_base) not in sys.path:
            sys.path.insert(0, str(_base))
        break

import _fixtures as fx
fx.bootstrap()

## Build the gate

`real_fold=True` so everything below uses genuine ViennaRNA structure predictions,
not `fx.StubFoldEngine`'s fake ones.

In [ ]:
host = fx.Host.ECOLI          # ECOLI | YEAST | HUMAN are all supported
gate = fx.toehold(host=host, real_fold=True)
fx.describe_gate(gate)        # available = True

## Inputs

A `TriggerSet` (the circuit's inputs) and a `Constraints` (the researcher's
limits). Both are made up here — override any field via keyword.

In [ ]:
triggers = fx.sample_trigger_set(n_activators=1)
constraints = fx.sample_constraints(max_switch_length=200)

for t in triggers.activators:
    print(f'{t.trigger_id}  {t.symbol:6}  {t.sequence}  {t.length} nt  GC {t.gc_content:.0f}%')
print('arity:', triggers.arity, '| logic:', triggers.logic_type)

## `required_tools()` — checked before a run starts

Implemented. A missing dependency should fail here, fast, not part-way through
an expensive run.

In [ ]:
fx.attempt('required_tools', gate.required_tools)

## `is_compatible()`

Cheap checks only, no folding: exactly `max_inputs` activators (a toehold has no
inverting mechanism, so no repressors), the host is supported, the trigger is long
enough to fit the shortest toehold plus a real stem, and even the smallest possible
switch fits `constraints.max_switch_length`.

In [ ]:
fx.attempt('is_compatible', lambda: gate.is_compatible(triggers, constraints))

## `generate_designs()`

Sweeps `toehold_lengths` against the trigger's full reverse complement — the
toehold/stem split is the only free parameter here (unlike `AntisenseNotGate`,
there is no separate window offset to sweep: the whole trigger is used every time).
No folding here — cheap, structural construction only.

In [ ]:
designs = fx.attempt(
    'generate_designs',
    lambda: list(gate.generate_designs(triggers, constraints)),
)

## `evaluate_design()`

Real ViennaRNA folding happens here: partition-function accessibility of the
initiation region (loop + AUG) in the switch alone (OFF — leakage) and in the
switch+trigger complex (ON — the state the trigger should unlock), plus the
hybridisation energy between switch and trigger.

In [ ]:
target = designs[0] if designs else fx.sample_design(gate, triggers)
fx.attempt('evaluate_design', lambda: gate.evaluate_design(target))

## `emit_sequence()` and `describe()` — output helpers

Called on a real generated design now that `generate_designs()` produces one,
rather than `fx.sample_design(...)`'s hand-built stand-in.

In [ ]:
design = designs[0] if designs else fx.sample_design(gate, triggers)
print('design_id  ', design.design_id)
print('gate_kind  ', design.gate_kind)
print('length     ', design.length, 'nt')
print('architecture:', design.architecture)
print('emit_sequence:', gate.emit_sequence(design))
print('describe     :', gate.describe(design))

## Folding engine

This notebook builds the gate with `real_fold=True`, so everything above already
used real ViennaRNA — the `evaluate_design()` numbers are genuine partition-function
predictions, not placeholders. For a faster (fake) run while just poking at
construction logic, drop `real_fold=True`:

```python
gate = fx.toehold(host=host)   # fx.StubFoldEngine, no ViennaRNA
```